[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C21_Frontier_Pretraining_Course/04_stability/04_stability.ipynb)

# 04 · 训练稳定性（模拟 spike 并演示各修法）

目标：用纯 numpy 模拟 loss spike，并从零实现各种稳定手段——**梯度裁剪、z-loss、warmup-decay 调度、bf16 精度模拟、spike 检测器**。

路线：梯度裁剪(核心) → 模拟 spike 看裁剪救场 → z-loss 控 logit → warmup-cosine 调度 → bf16 精度损失模拟 → spike 检测器 → ✏️ 练习(grad clip / z-loss / warmup-decay / spike 检测) → 📖 答案 → 🧪 真实 PaLM/OPT 稳定性算账胶囊。

> 心智模型：**裁剪管梯度爆炸、z-loss/QK-norm 管 logit 漂移、warmup 管冷启动、bf16+fp32主权重管精度、检测器管预警**。每个护栏治一种病。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
print('环境就绪，开始加护栏')

## 1 · 梯度裁剪：等比例缩放、只改大小不改方向（核心）

全局梯度范数超阈值 τ 时，整个梯度 ×`τ/‖g‖`。关键：**只改幅度、不改方向**。验证尖峰梯度被裁到 τ、方向不变。

In [ ]:
def clip_grad_norm(g, max_norm):
    '''全局梯度裁剪。返回 (裁剪后梯度, 裁剪前范数)。'''
    norm = np.linalg.norm(g)
    if norm > max_norm:
        g = g * (max_norm / norm)
    return g, norm

normal = rng.standard_normal(50) * 0.05    # 正常小梯度(范数<1)
spike = rng.standard_normal(50) * 30.0     # 爆炸梯度

for name, g in [('正常梯度', normal), ('尖峰梯度', spike)]:
    clipped, before = clip_grad_norm(g, max_norm=1.0)
    print(f'{name}: ‖g‖ {before:7.2f} -> 裁剪后 {np.linalg.norm(clipped):.4f}')

# 正常梯度(范数<1)不应被动
c_normal, _ = clip_grad_norm(normal, 1.0)
assert np.allclose(c_normal, normal), '范数未超阈值不应裁剪'
# 尖峰梯度应被裁到范数=1
c_spike, _ = clip_grad_norm(spike, 1.0)
assert np.isclose(np.linalg.norm(c_spike), 1.0), '尖峰应裁到范数=1'
# 方向不变
assert np.allclose(c_spike / np.linalg.norm(c_spike), spike / np.linalg.norm(spike))
print('\n✅ 梯度裁剪：尖峰被压到阈值、方向保留 —— 最基础的 spike 护栏')

## 2 · 模拟 loss spike：裁剪救场对比

模拟带**很高曲率**方向的损失(发散区)，中途注入一个**坏 batch**(巨大梯度)。对比有/无梯度裁剪：无裁剪 loss 在 spike 处暴涨甚至发散，有裁剪平稳越过。

(用一个 ill-conditioned 二次型：某方向曲率大到该 lr 已接近发散，spike 把它推过临界。)

In [ ]:
def optimize_spike(use_clip, steps=60, spike_step=30, lr=0.1, clip=1.0, seed=0):
    '''最小化 0.5*(a*w0^2 + w1^2)，a 大(病态)。spike_step 注入巨大梯度看 loss 轨迹。'''
    g = np.random.default_rng(seed)
    a = 19.0                                    # w0 方向曲率大；lr=0.1 下 |1-lr*a|=0.9 仍收敛但很敏感
    w = np.array([1.0, 5.0])
    losses = []
    for t in range(steps):
        grad = np.array([a * w[0], w[1]])       # 病态二次型的梯度
        if t == spike_step:
            grad = grad + g.standard_normal(2) * 2000.0  # 坏 batch：巨大梯度
        if use_clip:
            grad, _ = clip_grad_norm(grad, clip)
        w = w - lr * grad
        losses.append(0.5 * (a * w[0] ** 2 + w[1] ** 2))
    return np.array(losses)

no_clip = optimize_spike(use_clip=False)
with_clip = optimize_spike(use_clip=True)
print('spike 步(第30步)后的 loss:')
print(f'  无裁剪: 第31步 loss = {no_clip[31]:12.1f}  (暴涨!)')
print(f'  有裁剪: 第31步 loss = {with_clip[31]:12.4f}  (平稳)')
print(f'全程最大 loss: 无裁剪 {no_clip.max():.2e} | 有裁剪 {with_clip.max():.2e}')

# 无裁剪在 spike 处 loss 暴涨；有裁剪几乎不受影响
assert no_clip[31] > 100 * no_clip[29], '无裁剪应在 spike 后暴涨'
assert with_clip[31] < 2 * with_clip[29], '有裁剪应平稳越过 spike'
# 全程最大 loss：无裁剪因 spike 冲到极高，有裁剪始终受控
assert no_clip.max() > 50 * with_clip.max(), '无裁剪的峰值 loss 远高于有裁剪'
print('\n✅ 模拟证明：一个坏 batch 让无裁剪 loss 暴涨(病态方向尤危险)，裁剪让它平稳越过')

## 3 · z-loss：拉住输出 logit 的绝对量级

CE 损失只关心 logit 的**相对**大小(softmax 平移不变)，不约束**绝对**量级 → logit 可越漂越大。z-loss 加 `λ(logZ)²` 惩罚把 logit 拉回。

In [ ]:
def ce_loss(logits, target):
    m = logits.max()
    logZ = m + np.log(np.sum(np.exp(logits - m)))
    return logZ - logits[target], logZ

def ce_with_zloss(logits, target, zw=1e-4):
    ce, logZ = ce_loss(logits, target)
    return ce + zw * logZ ** 2, logZ, zw * logZ ** 2

# 两组 logit：相对关系相同，但绝对量级差 10 倍
small = np.array([1.0, 2.0, 3.0])
large = np.array([10.0, 20.0, 30.0])
ce_s, logZ_s = ce_loss(small, 2)
ce_l, logZ_l = ce_loss(large, 2)
print(f'small logits: CE={ce_s:.4f}, logZ={logZ_s:.2f}')
print(f'large logits: CE={ce_l:.4f}, logZ={logZ_l:.2f}')

# z-loss 对大 logit 的惩罚远大于小 logit
_, _, pen_s = ce_with_zloss(small, 2, zw=0.1)
_, _, pen_l = ce_with_zloss(large, 2, zw=0.1)
print(f'\nz-loss 惩罚: small {pen_s:.3f} vs large {pen_l:.3f}')
assert logZ_l > logZ_s, '大 logit 的 logZ 更大'
assert pen_l > 10 * pen_s, 'z-loss 对大 logit 惩罚远更重(拉它回来)'
print('✅ z-loss：惩罚随 logit 量级平方增长，把漂移的 logit 拉回合理范围')

## 4 · warmup + cosine 学习率调度

慢启动(warmup 防早期 spike) → 峰值 → 余弦衰减到≈0(精细收敛)。实现并验证三段形状。

In [ ]:
def lr_schedule(step, peak_lr, warmup_steps, total_steps):
    '''线性 warmup + 余弦衰减。'''
    if step < warmup_steps:
        return peak_lr * step / warmup_steps                    # 线性 warmup
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return peak_lr * 0.5 * (1 + np.cos(np.pi * progress))       # 余弦到 0

peak, warmup, total = 1e-3, 100, 1000
for s in [0, 25, 50, 100, 300, 700, 1000]:
    lr = lr_schedule(s, peak, warmup, total)
    phase = 'warmup' if s < warmup else 'decay'
    print(f'step {s:5d} [{phase}]: lr = {lr:.3e}')

assert lr_schedule(0, peak, warmup, total) == 0.0, 'warmup 从 0 起'
assert np.isclose(lr_schedule(warmup, peak, warmup, total), peak), 'warmup 末达峰值'
assert lr_schedule(total, peak, warmup, total) < 1e-9, '末端≈0'
assert lr_schedule(50, peak, warmup, total) < lr_schedule(100, peak, warmup, total), 'warmup 内上升'
assert lr_schedule(700, peak, warmup, total) < lr_schedule(300, peak, warmup, total), 'decay 内下降'
print('\n✅ warmup-cosine：0→峰值(warmup)→0(cosine)，三段形状正确')

## 5 · bf16 精度模拟：为什么需要 fp32 主权重

用截断尾数模拟 bf16(1符号/8指数/7尾数)。两个事实：① bf16 范围同 fp32(不溢出，胜 fp16)；② bf16 会**吞掉大权重上的小更新**(故需 fp32 主权重)。

In [ ]:
def to_bf16(x):
    '''模拟 bf16：把 float32 截断到 7 位尾数(round-to-nearest-even)。'''
    x = np.asarray(x, dtype=np.float32)
    u = x.view(np.uint32)
    bias = ((u >> 16) & 1) + np.uint32(0x7FFF)
    u = (u + bias) & np.uint32(0xFFFF0000)
    return u.view(np.float32)

# 事实1: bf16 范围大(同 fp32)，fp16 会溢出
big = np.float32(70000.0)
with np.errstate(over='ignore'):
    fp16_val = big.astype(np.float16)
print(f'70000 -> fp16: {float(fp16_val)} ({"溢出 inf!" if not np.isfinite(fp16_val) else "ok"})')
print(f'70000 -> bf16: {float(to_bf16(big))} (bf16 范围同 fp32，不溢出)')
assert not np.isfinite(fp16_val), 'fp16 应溢出'
assert np.isfinite(to_bf16(big)), 'bf16 不溢出'

# 事实2: bf16 吞掉大权重上的小更新
w = np.float32(1000.0); small_update = np.float32(0.01)
w_bf16 = to_bf16(to_bf16(w) - to_bf16(small_update))
w_fp32 = w - small_update
print(f'\n1000 - 0.01: fp32={w_fp32} (变了) | bf16={float(w_bf16)} (没变，更新被吞!)')
assert w_fp32 != w, 'fp32 留住了小更新'
assert w_bf16 == to_bf16(w), 'bf16 吞掉了小更新'
print('✅ bf16 范围大不溢出(胜 fp16)，但吞小更新 -> 必须配 fp32 主权重累加')

## 6 · spike 检测器：grad norm 是先行预警

grad norm 的突然飙升常**先于** loss spike。实现一个基于「相对历史中位数」的检测器，在 grad norm 异常时报警。

In [ ]:
def detect_spike(grad_norms, window=10, factor=3.0):
    '''若当前 grad norm > factor × 最近 window 步的中位数，报警(返回触发的步索引)。'''
    alerts = []
    for t in range(window, len(grad_norms)):
        recent_median = np.median(grad_norms[t - window:t])
        if grad_norms[t] > factor * recent_median:
            alerts.append(t)
    return alerts

# 模拟一段 grad norm：平稳 + 第40步一个尖峰
g = np.random.default_rng(3)
gn = np.abs(g.standard_normal(60) * 0.1 + 1.0)   # 平稳在 ~1.0
gn[40] = 8.0                                       # 注入 grad norm 尖峰
alerts = detect_spike(gn, window=10, factor=3.0)
print(f'grad norm 尖峰在第 40 步; 检测器报警于: {alerts}')
assert 40 in alerts, '应检测到第40步的尖峰'
# 平稳段不应误报
assert len([a for a in alerts if a != 40]) == 0, '平稳段不应误报'
print('✅ spike 检测器：grad norm 异常飙升时报警(给回滚/跳数据争取预警窗口)')

---
## ✏️ 练习 1：带返回裁剪比例的梯度裁剪

实现 `clip_with_ratio(g, max_norm)`：返回 `(裁剪后梯度, 实际缩放比例)`。比例 = `min(1, max_norm/‖g‖)`(未裁剪时为 1.0)。

In [ ]:
def clip_with_ratio(g, max_norm):
    # TODO: norm=‖g‖; ratio=min(1, max_norm/norm); 返回 (g*ratio, ratio)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
g_big = np.ones(4) * 10.0       # 范数 = 20
g_small = np.ones(4) * 0.1      # 范数 = 0.2
_, r_big = clip_with_ratio(g_big, 1.0)
out_small, r_small = clip_with_ratio(g_small, 1.0)
print(f'大梯度缩放比例 {r_big:.4f}, 小梯度比例 {r_small:.4f}')
assert abs(r_big - 1.0/20.0) < 1e-9, '范数20裁到1 -> 比例 1/20'
assert r_small == 1.0, '小梯度不裁 -> 比例 1.0'
assert np.allclose(out_small, g_small), '未裁剪应原样返回'
clipped, _ = clip_with_ratio(g_big, 1.0)
assert np.isclose(np.linalg.norm(clipped), 1.0)
print('✅ 练习 1 通过：裁剪 + 返回缩放比例(可用于监控裁剪频率)')

## ✏️ 练习 2：z-loss 项

实现 `zloss_term(logits, weight)`：返回 `weight * (logZ)²`，其中 `logZ = logsumexp(logits)`(数值稳定：先减最大值)。

In [ ]:
def zloss_term(logits, weight):
    # TODO: m=max(logits); logZ=m+log(sum(exp(logits-m))); 返回 weight*logZ^2
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
small = np.array([1.0, 2.0, 3.0])
large = np.array([100.0, 200.0, 300.0])
z_s = zloss_term(small, 1e-4)
z_l = zloss_term(large, 1e-4)
print(f'z-loss: small={z_s:.6f}, large={z_l:.4f}')
assert z_l > z_s, '大 logit z-loss 更大'
# 数值稳定：超大 logit 不应 overflow
huge = np.array([1000.0, 2000.0, 3000.0])
z_huge = zloss_term(huge, 1e-4)
assert np.isfinite(z_huge), '应数值稳定(先减最大值)，不 overflow'
# logZ 应≈最大 logit(当 logits 差距大时)
assert abs(np.sqrt(z_s/1e-4) - 3.0) < 0.5, 'small 的 logZ 应≈3(最大 logit 附近)'
print('✅ 练习 2 通过：z-loss 数值稳定且随 logit 量级增长')

## ✏️ 练习 3：WSD 调度

实现 `wsd_schedule(step, peak, warmup, stable_end, total)`：
- `step<warmup`：线性 warmup 到 peak；
- `warmup<=step<stable_end`：恒定 peak；
- `step>=stable_end`：线性衰减到 0。

(WSD = Warmup-Stable-Decay，可中途加数据。)

In [ ]:
def wsd_schedule(step, peak, warmup, stable_end, total):
    # TODO: 三段：warmup 线性升 / stable 恒定 / decay 线性降到 0
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
peak, wu, se, tot = 1e-3, 100, 800, 1000
assert wsd_schedule(0, peak, wu, se, tot) == 0.0
assert np.isclose(wsd_schedule(wu, peak, wu, se, tot), peak), 'warmup 末达峰'
assert np.isclose(wsd_schedule(400, peak, wu, se, tot), peak), 'stable 段恒定'
assert np.isclose(wsd_schedule(se, peak, wu, se, tot), peak), 'stable 末仍是峰'
assert wsd_schedule(900, peak, wu, se, tot) < peak, 'decay 段下降'
assert wsd_schedule(tot, peak, wu, se, tot) < 1e-9, '末端≈0'
# stable 段确实恒定
assert wsd_schedule(200, peak, wu, se, tot) == wsd_schedule(600, peak, wu, se, tot)
print('✅ 练习 3 通过：WSD 三段(升/恒定/降)，中间恒定段支持灵活延长训练')

## ✏️ 练习 4：基于绝对 + 相对的 spike 检测

实现 `robust_spike_detector(losses, abs_thresh, rel_factor)`：报警条件为
**(loss > abs_thresh 的绝对阈值) 或 (loss > rel_factor × 前一步 loss 的相对暴涨)**。返回报警步索引列表(从第1步起)。

In [ ]:
def robust_spike_detector(losses, abs_thresh, rel_factor):
    # TODO: for t in 1..len: if losses[t]>abs_thresh or losses[t]>rel_factor*losses[t-1]: 记录 t
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
losses = [2.0, 1.9, 1.85, 1.8, 9.0, 1.82, 1.8, 50.0]   # 第4步相对暴涨, 第7步绝对超标
alerts = robust_spike_detector(losses, abs_thresh=20.0, rel_factor=3.0)
print(f'报警步: {alerts}')
assert 4 in alerts, '第4步 loss 1.8->9.0(相对暴涨)应报警'
assert 7 in alerts, '第7步 loss 50>20(绝对超标)应报警'
assert 1 not in alerts and 5 not in alerts, '正常步不报警'
print('✅ 练习 4 通过：绝对+相对双判据 spike 检测(漏报更少)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def clip_with_ratio(g, max_norm):
    norm = np.linalg.norm(g)
    ratio = min(1.0, max_norm / norm) if norm > 0 else 1.0
    return g * ratio, ratio

In [ ]:
# 练习 2 参考答案
def zloss_term(logits, weight):
    m = logits.max()
    logZ = m + np.log(np.sum(np.exp(logits - m)))
    return weight * logZ ** 2

In [ ]:
# 练习 3 参考答案
def wsd_schedule(step, peak, warmup, stable_end, total):
    if step < warmup:
        return peak * step / warmup
    if step < stable_end:
        return peak
    return peak * max(0.0, (total - step) / (total - stable_end))

In [ ]:
# 练习 4 参考答案
def robust_spike_detector(losses, abs_thresh, rel_factor):
    alerts = []
    for t in range(1, len(losses)):
        if losses[t] > abs_thresh or losses[t] > rel_factor * losses[t-1]:
            alerts.append(t)
    return alerts

---
## 🧪 真实数据胶囊：checkpoint 频率 vs spike 损失算账

spike 不可逆，唯一退路是回滚到 checkpoint。存得勤 -> IO 开销；存得稀 -> spike 时损失更多进度。用真实量级算这个权衡：一次大训练的最优 checkpoint 频率。

In [ ]:
def expected_waste(ckpt_interval_steps, spike_rate_per_step, total_steps, step_cost_usd):
    '''粗略期望浪费(美元)：每次 spike 平均回滚半个 checkpoint 区间的进度。'''
    n_spikes = spike_rate_per_step * total_steps
    wasted_steps_per_spike = ckpt_interval_steps / 2     # 平均回滚半个区间
    return n_spikes * wasted_steps_per_spike * step_cost_usd

# 真实量级：100万步训练，每步 $50，spike 率 1e-5(约每10万步一次)
total_steps = 1_000_000
step_cost = 50.0
spike_rate = 1e-5
print(f"{'ckpt间隔':>10} {'存储开销':>12} {'spike浪费':>14} {'总成本':>14}")
for interval in [100, 500, 2000, 10000]:
    # 存储开销：粗略正比于 存储次数(每次假设 $20)
    n_ckpts = total_steps / interval
    storage = n_ckpts * 20
    waste = expected_waste(interval, spike_rate, total_steps, step_cost)
    print(f'{interval:10d} {storage:12.0f} {waste:14.0f} {storage+waste:14.0f}')

# 间隔太大 -> spike 浪费暴涨；间隔太小 -> 存储开销暴涨。存在最优
w_dense = expected_waste(100, spike_rate, total_steps, step_cost)
w_sparse = expected_waste(10000, spike_rate, total_steps, step_cost)
assert w_sparse > w_dense, '存得稀 -> spike 浪费更多'
print('\n✅ 胶囊：checkpoint 频率是「存储开销 vs spike 进度损失」的权衡，存在最优间隔')

**🧪 胶囊练习**：实现 `spike_waste(n_spikes, ckpt_interval, step_cost)`：n 次 spike，每次平均回滚半个 checkpoint 区间，返回总浪费美元。

In [ ]:
def spike_waste(n_spikes, ckpt_interval, step_cost):
    # TODO: n_spikes * (ckpt_interval/2) * step_cost
    raise NotImplementedError

In [ ]:
# 自测
w = spike_waste(10, 2000, 50.0)
print(f'10 次 spike, 2000 步间隔, $50/步 -> 浪费 ${w:,.0f}')
assert w == 10 * 1000 * 50.0
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def spike_waste(n_spikes, ckpt_interval, step_cost):
    return n_spikes * (ckpt_interval / 2) * step_cost

### 小结
- **梯度裁剪**：范数超阈值就等比缩放(只改幅度不改方向)，挡住单次梯度爆炸——最基础护栏。
- **z-loss / QK-norm**：CE 不管 logit 绝对量级，z-loss 软惩罚输出 logZ、QK-norm 硬归一化注意力 logit，打断「logit 漂移→softmax 饱和」正反馈。
- **warmup-decay**：冷启动慢升(防早期 spike)、峰值快跑、衰减精收敛；WSD 中间恒定段支持灵活延长。
- **bf16+fp32 主权重**：bf16 范围同 fp32(不溢出，胜 fp16)但吞小更新，故必须 fp32 主权重累加。
- **初始化/深度缩放/Pre-LN** 是「天生稳定」地基；**检测器+频繁 checkpoint+回滚跳数据** 是应急流程。

下一站：**模块 05 · 数据配比与合成数据** —— 数据干净、训练稳了，最后一问：各 domain 按什么比例喂？能不能用模型造数据？